# Sensor Rankings
## FloodNet NYC Tutorial
Author: Mark Bauer

# Summary
TBD

# Table of Contents
TBD

# Import Libraries

In [1]:
# libraries
import numpy as np
import pandas as pd
import polars as pl
import duckdb
from typing import Iterable, Optional
import ast

# Load and Inspect Data
For a more robust approach to inspecting the data, please refer to notebook 01-load-inspect.ipynb.

In [2]:
# list files in data directory
%ls data

Flood Events_Data Dictionary.xlsx
Floodnet_Open_Data_-_Data_Description_03.05.2026.pdf
flood-events.csv
metadata.csv


In [3]:
# read in flood sensor deployments as dataframe
metadata_path = "data/metadata.csv"
metadata_df = pl.read_csv(metadata_path, try_parse_dates=True)

# inspect the shape of the data
n_rows, n_columns = metadata_df.shape
print("Metadata:")
print(f"Number of rows: {n_rows}")
print(f"Number of columns: {n_columns}")

# read in flood events as dataframe
events_path = "data/flood-events.csv"
events_df = pl.read_csv(events_path, infer_schema_length=100000, try_parse_dates=True)

# inspect the shape of the data
n_rows, n_columns = events_df.shape
print("\nFlood Events:")
print(f"Number of rows: {n_rows:,}")
print(f"Number of columns: {n_columns}")

Metadata:
Number of rows: 371
Number of columns: 16

Flood Events:
Number of rows: 1,887
Number of columns: 13


# Join Sensor Metadata and Flood Events Datasets

In [4]:
# merge flood sensor events with sensor metadata
merged_df = events_df.join(
    metadata_df.drop("sensor_name"),
    on="sensor_id",
    how="left",  # keep all events, even if metadata is missing
)

# inspect the shape of the merged data
n_rows, n_columns = merged_df.shape
print(f"Number of rows of merged df: {n_rows:,}")
print(f"Number of columns: {n_columns}")

# events without matching metadata
missing_meta_count = merged_df["sensor_id"].null_count()
print(f"Events without matching sensor metadata: {missing_meta_count}\n")

# preview the first few rows
merged_df.head()

Number of rows of merged df: 1,887
Number of columns: 27
Events without matching sensor metadata: 0



sensor_name,sensor_id,flood_start_time,flood_end_time,max_depth_inches,onset_time_mins,drain_time_mins,duration_mins,duration_above_4_inches_mins,duration_above_12_inches_mins,duration_above_24_inches_mins,flood_profile_depth_inches,flood_profile_time_secs,date_installed,tidally_influenced,date_removed,street_name,borough,zipcode,community_board,council_district,census_tract,nta,latitude,longitude,lowest_point_height_delta_inches,location
str,str,datetime[μs],datetime[μs],f64,f64,f64,f64,f64,f64,f64,str,str,datetime[μs],str,datetime[μs],str,str,i64,i64,str,i64,str,f64,f64,f64,str
"""Q - Beach 84 St""","""Q-beach-84-st-0me680""",2023-10-30 12:00:39,2023-10-30 16:38:19,17.72,133.86,143.81,277.66,211.41,103.6,0.0,"""[0.00, 0.87, 1.02, 1.38, 1.93,…","""[0, 1008, 1134, 1512, 1826, 20…",2021-12-10 00:00:00,"""Yes""",null,"""Beach 84th Street""","""Queens""",11693,414,"""QN14""",4094202,"""QN1402""",40.59136,-73.80996,8.27,"""POINT (-73.80996 40.59136)"""
"""BX - 1st St/Avenue A""","""BX-1st-st-avenue-a-1zby90""",2025-04-26 23:17:20,2025-04-27 11:55:34,6.18,27.95,730.29,758.24,132.81,0.0,0.0,"""[0.00, 0.71, 1.38, 1.38, 1.38,…","""[0, 64, 129, 132, 196, 260, 32…",2024-07-18 00:00:00,"""No""",null,"""Avenue A""","""Bronx""",10474,202,"""BX02""",2011702,"""BX0201""",40.812851,-73.881043,3.62,"""POINT (-73.881043 40.812851)"""
"""Q - Beach 35 St/Beach Channel …","""Q-beach-35-st-beach-channel-dr…",2024-02-09 11:41:01,2024-02-09 13:20:57,5.31,54.51,45.43,99.94,45.09,0.0,0.0,"""[0.00, 0.43, 0.63, 0.71, 0.94,…","""[0, 68, 136, 204, 273, 341, 40…",2023-12-19 00:00:00,"""Yes""",null,"""Beach 35th Street""","""Queens""",11691,414,"""QN14""",4099200,"""QN1401""",40.596618,-73.767808,6.97,"""POINT (-73.767808 40.596618)"""
"""BX - Tibbett Ave/W 234th St""","""BX-w-234th-st-tibbett-ave-2cak…",2025-08-20 21:05:19,2025-08-21 00:46:19,1.18,80.0,141.0,221.0,0.0,0.0,0.0,"""[0.00, 0.43, 0.47, 0.47, 0.51,…","""[0, 60, 120, 180, 240, 300, 36…",2025-03-27 00:00:00,"""No""",null,"""Tibbett Avenue""","""Bronx""",10463,208,"""BX08""",2028700,"""BX0802""",40.88326,-73.90585,4.09,"""POINT (-73.90585 40.88326)"""
"""Q - Davenport Ct 1""","""Q-davenport-ct-1-07zks0""",2024-01-10 00:39:25,2024-01-10 02:30:40,2.09,59.82,51.43,111.25,0.0,0.0,0.0,"""[0.00, 0.47, 0.47, 0.43, 0.43,…","""[0, 63, 127, 189, 252, 315, 37…",2021-03-05 00:00:00,"""Yes""",null,"""Davenport Court""","""Queens""",11414,410,"""QN10""",4088400,"""QN1003""",40.653387,-73.830559,6.97,"""POINT (-73.830559 40.653387)"""


In [5]:
# inspect column information and data types
merged_df.schema

Schema([('sensor_name', String),
        ('sensor_id', String),
        ('flood_start_time', Datetime(time_unit='us', time_zone=None)),
        ('flood_end_time', Datetime(time_unit='us', time_zone=None)),
        ('max_depth_inches', Float64),
        ('onset_time_mins', Float64),
        ('drain_time_mins', Float64),
        ('duration_mins', Float64),
        ('duration_above_4_inches_mins', Float64),
        ('duration_above_12_inches_mins', Float64),
        ('duration_above_24_inches_mins', Float64),
        ('flood_profile_depth_inches', String),
        ('flood_profile_time_secs', String),
        ('date_installed', Datetime(time_unit='us', time_zone=None)),
        ('tidally_influenced', String),
        ('date_removed', Datetime(time_unit='us', time_zone=None)),
        ('street_name', String),
        ('borough', String),
        ('zipcode', Int64),
        ('community_board', Int64),
        ('council_district', String),
        ('census_tract', Int64),
        ('nta', Stri

# Data Cleaning
Converting dates to datetime format.

In [6]:
# convert date columns to datetime
merged_df = merged_df.with_columns(
    pl.col("flood_start_time")
        .dt.convert_time_zone(time_zone='America/New_York')
        .alias("flood_start_time_est"),
    pl.col("flood_end_time")
        .dt.convert_time_zone(time_zone='America/New_York')
        .alias("flood_end_time_est")
)

# preview new dtypes
cols =  [
    'flood_start_time',
    'flood_end_time',
    'flood_start_time_est',
    'flood_end_time_est',
]

merged_df.select(cols).head()

flood_start_time,flood_end_time,flood_start_time_est,flood_end_time_est
datetime[μs],datetime[μs],"datetime[μs, America/New_York]","datetime[μs, America/New_York]"
2023-10-30 12:00:39,2023-10-30 16:38:19,2023-10-30 08:00:39 EDT,2023-10-30 12:38:19 EDT
2025-04-26 23:17:20,2025-04-27 11:55:34,2025-04-26 19:17:20 EDT,2025-04-27 07:55:34 EDT
2024-02-09 11:41:01,2024-02-09 13:20:57,2024-02-09 06:41:01 EST,2024-02-09 08:20:57 EST
2025-08-20 21:05:19,2025-08-21 00:46:19,2025-08-20 17:05:19 EDT,2025-08-20 20:46:19 EDT
2024-01-10 00:39:25,2024-01-10 02:30:40,2024-01-09 19:39:25 EST,2024-01-09 21:30:40 EST


# Sensor Rankings

# 1) Max Flood Depth
## 1.1 Top 10 Flood Events by Maximum Flood Depth. 
Rankings can have repeated sensors, as table is ranked by events.

In [7]:
sql = """
    SELECT
        sensor_name,
        flood_start_time_est,
        flood_end_time_est,
        max_depth_inches,
        duration_mins,
        tidally_influenced,
        street_name,
        borough
    FROM merged_df
    ORDER BY max_depth_inches DESC
    LIMIT 10
"""

duckdb.sql(sql).pl()

sensor_name,flood_start_time_est,flood_end_time_est,max_depth_inches,duration_mins,tidally_influenced,street_name,borough
str,"datetime[μs, America/New_York]","datetime[μs, America/New_York]",f64,f64,str,str,str
"""Q - Davenport Ct 1""",2022-12-23 05:24:23 EST,2022-12-23 16:29:21 EST,40.87,664.96,"""Yes""","""Davenport Court""","""Queens"""
"""Q - Beach 84 St""",2022-12-23 05:00:52 EST,2022-12-23 11:19:11 EST,38.19,378.31,"""Yes""","""Beach 84th Street""","""Queens"""
"""BK - Carroll St/4th Av""",2023-09-29 07:53:39 EDT,2023-09-29 10:02:50 EDT,35.94,129.18,"""No""","""Carroll Street""","""Brooklyn"""
"""BK - Carroll St/4th Av""",2021-09-01 21:27:16 EDT,2021-09-01 23:11:26 EDT,35.59,104.17,"""No""","""Carroll Street""","""Brooklyn"""
"""Q - Davenport Ct 1""",2024-01-13 08:05:35 EST,2024-01-13 17:19:03 EST,34.69,553.46,"""Yes""","""Davenport Court""","""Queens"""
"""BX - Ditmars St/Hunter Ave 2""",2024-04-03 12:04:19 EDT,2024-04-04 02:34:36 EDT,34.65,870.27,"""Yes""","""Ditmars Street""","""Bronx"""
"""Q - Beach 84 St""",2024-01-13 06:59:39 EST,2024-01-13 12:48:17 EST,32.52,348.63,"""Yes""","""Beach 84th Street""","""Queens"""
"""Q - Russell St 2""",2022-12-23 05:12:13 EST,2022-12-23 11:34:35 EST,31.77,382.38,"""Yes""","""Russell Street""","""Queens"""
"""Q - Russell St 1""",2022-12-23 05:10:48 EST,2022-12-23 11:04:16 EST,31.34,353.47,"""Yes""","""Russell Street""","""Queens"""


## 1.2 Top 10 Sensors by Maximum Flood Depth.
Table is ranked by depth per unique sensor. We take the greatest depth per sensor.

In [8]:
sql = """
    SELECT
        DISTINCT ON(sensor_name)
        sensor_name,
        flood_start_time_est,
        flood_end_time_est,
        max_depth_inches,
        duration_mins,
        tidally_influenced,
        street_name,
        borough
    FROM merged_df
    ORDER BY max_depth_inches DESC
    LIMIT 10
"""

duckdb.sql(sql).pl()

sensor_name,flood_start_time_est,flood_end_time_est,max_depth_inches,duration_mins,tidally_influenced,street_name,borough
str,"datetime[μs, America/New_York]","datetime[μs, America/New_York]",f64,f64,str,str,str
"""Q - Davenport Ct 1""",2022-12-23 05:24:23 EST,2022-12-23 16:29:21 EST,40.87,664.96,"""Yes""","""Davenport Court""","""Queens"""
"""Q - Beach 84 St""",2022-12-23 05:00:52 EST,2022-12-23 11:19:11 EST,38.19,378.31,"""Yes""","""Beach 84th Street""","""Queens"""
"""BK - Carroll St/4th Av""",2023-09-29 07:53:39 EDT,2023-09-29 10:02:50 EDT,35.94,129.18,"""No""","""Carroll Street""","""Brooklyn"""
"""BX - Ditmars St/Hunter Ave 2""",2024-04-03 12:04:19 EDT,2024-04-04 02:34:36 EDT,34.65,870.27,"""Yes""","""Ditmars Street""","""Bronx"""
"""Q - Russell St 2""",2022-12-23 05:12:13 EST,2022-12-23 11:34:35 EST,31.77,382.38,"""Yes""","""Russell Street""","""Queens"""
"""Q - Russell St 1""",2022-12-23 05:10:48 EST,2022-12-23 11:04:16 EST,31.34,353.47,"""Yes""","""Russell Street""","""Queens"""
"""BK - 9th St/Smith St""",2021-09-01 20:11:58 EDT,2021-09-02 00:42:32 EDT,27.76,270.57,"""No""","""9th Street""","""Brooklyn"""
"""BK - Wallabout St/Throop Ave""",2023-09-29 08:05:49 EDT,2023-09-29 11:03:49 EDT,25.08,178.0,"""No""","""Wallabout Street""","""Brooklyn"""
"""BK - Lee Ave/Middleton St""",2023-09-29 08:07:18 EDT,2023-09-29 12:12:42 EDT,23.54,245.39,"""No""","""Lee Avenue""","""Brooklyn"""


## 1.3 Top 10 Tidally Influenced Sensors by Maximum Flood Depth.

In [9]:
sql = """
    SELECT
        DISTINCT ON(sensor_name) 
        sensor_name,
        flood_start_time_est,
        flood_end_time_est,
        max_depth_inches,
        duration_mins,
        tidally_influenced,
        street_name,
        borough
    FROM merged_df
    WHERE tidally_influenced == 'Yes'
    ORDER BY max_depth_inches DESC
    LIMIT 10
"""

duckdb.sql(sql).pl()

sensor_name,flood_start_time_est,flood_end_time_est,max_depth_inches,duration_mins,tidally_influenced,street_name,borough
str,"datetime[μs, America/New_York]","datetime[μs, America/New_York]",f64,f64,str,str,str
"""Q - Davenport Ct 1""",2022-12-23 05:24:23 EST,2022-12-23 16:29:21 EST,40.87,664.96,"""Yes""","""Davenport Court""","""Queens"""
"""Q - Beach 84 St""",2022-12-23 05:00:52 EST,2022-12-23 11:19:11 EST,38.19,378.31,"""Yes""","""Beach 84th Street""","""Queens"""
"""BX - Ditmars St/Hunter Ave 2""",2024-04-03 12:04:19 EDT,2024-04-04 02:34:36 EDT,34.65,870.27,"""Yes""","""Ditmars Street""","""Bronx"""
"""Q - Russell St 2""",2022-12-23 05:12:13 EST,2022-12-23 11:34:35 EST,31.77,382.38,"""Yes""","""Russell Street""","""Queens"""
"""Q - Russell St 1""",2022-12-23 05:10:48 EST,2022-12-23 11:04:16 EST,31.34,353.47,"""Yes""","""Russell Street""","""Queens"""
"""Q - Beach 43rd St""",2022-12-23 06:07:36 EST,2022-12-23 09:57:30 EST,23.43,229.91,"""Yes""","""Beach 43rd Street""","""Queens"""
"""Q - 102nd St/160th Ave""",2024-01-13 07:46:45 EST,2024-01-13 12:18:31 EST,21.69,271.78,"""Yes""","""102nd Street""","""Queens"""
"""Q - Beach 72nd St/Almeda Ave""",2024-01-13 09:15:14 EST,2024-01-13 16:52:32 EST,21.5,457.29,"""Yes""","""Beach 72nd Street""","""Queens"""
"""Q - Beach 84 St (2)""",2025-10-30 13:42:55 EDT,2025-10-30 18:33:54 EDT,20.43,290.98,"""Yes""","""Beach 84th Street""","""Queens"""


## 1.4 Top 10 Non-Tidally Influenced Sensors by Maximum Flood Depth.
Non-tidally influenced sensors are sometimes evaluated independently, since their flood depths are driven primarily by rainfall and drainage conditions rather than tidal cycles, resulting in different flooding dynamics than tidally influenced sensors. This is visible if the frequency of flooding events.

In [10]:
sql = """
    SELECT
        DISTINCT ON(sensor_name) 
        sensor_name,
        flood_start_time_est,
        flood_end_time_est,
        max_depth_inches,
        duration_mins,
        tidally_influenced,
        street_name,
        borough
    FROM merged_df
    WHERE tidally_influenced == 'No'
    ORDER BY max_depth_inches DESC
    LIMIT 10
"""

duckdb.sql(sql).pl()

sensor_name,flood_start_time_est,flood_end_time_est,max_depth_inches,duration_mins,tidally_influenced,street_name,borough
str,"datetime[μs, America/New_York]","datetime[μs, America/New_York]",f64,f64,str,str,str
"""BK - Carroll St/4th Av""",2023-09-29 07:53:39 EDT,2023-09-29 10:02:50 EDT,35.94,129.18,"""No""","""Carroll Street""","""Brooklyn"""
"""BK - 9th St/Smith St""",2021-09-01 20:11:58 EDT,2021-09-02 00:42:32 EDT,27.76,270.57,"""No""","""9th Street""","""Brooklyn"""
"""BK - Wallabout St/Throop Ave""",2023-09-29 08:05:49 EDT,2023-09-29 11:03:49 EDT,25.08,178.0,"""No""","""Wallabout Street""","""Brooklyn"""
"""BK - Lee Ave/Middleton St""",2023-09-29 08:07:18 EDT,2023-09-29 12:12:42 EDT,23.54,245.39,"""No""","""Lee Avenue""","""Brooklyn"""
"""BK - Kingston Ave/Rutland Rd""",2025-10-30 15:02:07 EDT,2025-10-30 17:26:22 EDT,23.35,144.24,"""No""","""Kingston Avenue""","""Brooklyn"""
"""Q - 204th St/ 100th Ave""",2025-10-30 16:04:44 EDT,2025-10-30 16:47:45 EDT,23.35,43.02,"""No""","""204th Street""","""Queens"""
"""BK - Knickerbocker Ave/ Hart S…",2025-10-30 15:09:13 EDT,2025-10-30 17:15:42 EDT,22.76,126.48,"""No""","""Knickerbocker Avenue""","""Brooklyn"""
"""Q - Peck Ave/153rd st""",2025-10-30 15:54:42 EDT,2025-10-30 17:40:42 EDT,22.64,106.0,"""No""","""153rd Street""","""Queens"""
"""Q - 56th Ave/ Springfield Blvd""",2025-10-30 15:51:23 EDT,2025-10-30 19:30:22 EDT,21.93,218.98,"""No""","""56th Avenue""","""Queens"""


# 2) Frequency of Flood Events
Counts are normalized by the number of days each sensor was deployed. As sensors were installed in different years, raw event counts are not directly comparable without normalization.

## Normalize Event Counts

In [11]:
deployment_days = duckdb.sql("""
    SELECT
        sensor_id,
        date_diff('day', date_installed, current_date()) AS deployment_days
    FROM metadata_df
""")

flood_events = duckdb.sql("""
    SELECT
        sensor_id,
        sensor_name,
        street_name,
        borough,
        tidally_influenced,
        COUNT(*) AS flood_events,
    FROM merged_df
    GROUP BY ALL
""")

sql = """
    SELECT
        *,
        ROUND(flood_events /  deployment_days * (365 / 12.), 2) AS flood_events_per_month
    FROM flood_events
    JOIN deployment_days USING (sensor_id)
    ORDER BY flood_events_per_month DESC
    LIMIT 10
"""

duckdb.sql(sql).pl()

sensor_id,sensor_name,street_name,borough,tidally_influenced,flood_events,deployment_days,flood_events_per_month
str,str,str,str,str,i64,i64,f64
"""BX-ditmars-st-hunter-ave-1kwrk…","""BX - Ditmars St/Hunter Ave 2""","""Ditmars Street""","""Bronx""","""Yes""",288,879,9.97
"""Q-beach-84-st-0me680""","""Q - Beach 84 St""","""Beach 84th Street""","""Queens""","""Yes""",376,1550,7.38
"""Q-brookville-blvd-rockaway-blv…","""Q - Brookville Blvd/ Snake Rd …","""Brookville Boulevard""","""Queens""","""Yes""",79,515,4.67
"""Q-beach-84th-st-beach-channel-…","""Q - Beach 84 St (2)""","""Beach 84th Street""","""Queens""","""Yes""",16,131,3.72
"""Q-brookville-blvd-rockaway-blv…","""Q - Brookville Blvd/ Snake Rd …","""Brookville Boulevard""","""Queens""","""Yes""",52,515,3.07
"""Q-brookville-blvd-149th-ave-3-…","""Q - Brookville Blvd/ Snake Rd …","""Brookville Boulevard""","""Queens""","""Yes""",42,515,2.48
"""Q-beach-35-st-beach-channel-dr…","""Q - Beach 35 St/Beach Channel …","""Beach 35th Street""","""Queens""","""Yes""",58,811,2.18
"""Q-russell-st-1-07zqc0""","""Q - Russell St 1""","""Russell Street""","""Queens""","""Yes""",125,1830,2.08
"""Q-102nd-st-160th-ave-1a5ng0""","""Q - 102nd St/160th Ave""","""102nd Street""","""Queens""","""Yes""",71,1088,1.98


## 2.1 Top 10 Highest Number of Events by Tidally Influenced Sensors.

In [12]:
sql = """
    SELECT
        *,
        ROUND(flood_events / deployment_days * (365 / 12.), 2) AS flood_events_per_month
    FROM flood_events
    JOIN deployment_days USING (sensor_id)
    WHERE tidally_influenced == 'Yes'
    ORDER BY flood_events_per_month DESC
    LIMIT 10
"""

duckdb.sql(sql).pl()

sensor_id,sensor_name,street_name,borough,tidally_influenced,flood_events,deployment_days,flood_events_per_month
str,str,str,str,str,i64,i64,f64
"""BX-ditmars-st-hunter-ave-1kwrk…","""BX - Ditmars St/Hunter Ave 2""","""Ditmars Street""","""Bronx""","""Yes""",288,879,9.97
"""Q-beach-84-st-0me680""","""Q - Beach 84 St""","""Beach 84th Street""","""Queens""","""Yes""",376,1550,7.38
"""Q-brookville-blvd-rockaway-blv…","""Q - Brookville Blvd/ Snake Rd …","""Brookville Boulevard""","""Queens""","""Yes""",79,515,4.67
"""Q-beach-84th-st-beach-channel-…","""Q - Beach 84 St (2)""","""Beach 84th Street""","""Queens""","""Yes""",16,131,3.72
"""Q-brookville-blvd-rockaway-blv…","""Q - Brookville Blvd/ Snake Rd …","""Brookville Boulevard""","""Queens""","""Yes""",52,515,3.07
"""Q-brookville-blvd-149th-ave-3-…","""Q - Brookville Blvd/ Snake Rd …","""Brookville Boulevard""","""Queens""","""Yes""",42,515,2.48
"""Q-beach-35-st-beach-channel-dr…","""Q - Beach 35 St/Beach Channel …","""Beach 35th Street""","""Queens""","""Yes""",58,811,2.18
"""Q-russell-st-1-07zqc0""","""Q - Russell St 1""","""Russell Street""","""Queens""","""Yes""",125,1830,2.08
"""Q-102nd-st-160th-ave-1a5ng0""","""Q - 102nd St/160th Ave""","""102nd Street""","""Queens""","""Yes""",71,1088,1.98


## 2.2 Highest Number of Events by Non-Tidally Influenced Sensors.

In [13]:
sql = """
    SELECT
        *,
        ROUND(flood_events / deployment_days * (365 / 12.), 2) AS flood_events_per_month
    FROM flood_events
    JOIN deployment_days USING (sensor_id)
    WHERE tidally_influenced = 'No'
    ORDER BY flood_events_per_month DESC
    LIMIT 10
"""

duckdb.sql(sql).pl()

sensor_id,sensor_name,street_name,borough,tidally_influenced,flood_events,deployment_days,flood_events_per_month
str,str,str,str,str,i64,i64,f64
"""BX-w-234th-st-tibbett-ave-2cak…","""BX - Tibbett Ave/W 234th St""","""Tibbett Avenue""","""Bronx""","""No""",11,347,0.96
"""BX-webster-ave-e-166th-st-2151…","""BX - Webster Ave/E 166th St""","""Webster Avenue""","""Bronx""","""No""",15,564,0.81
"""BK-neptune-ave-w-35th-st-1xvfk…","""BK - Neptune Ave W 35th St""","""West 35th Street""","""Brooklyn""","""No""",12,627,0.58
"""BK-wythe-ave-wallabout-st-19qz…","""BK - Wythe Ave/Wallabout St""","""Wythe Avenue""","""Brooklyn""","""No""",19,1096,0.53
"""Q-56th-ave-springfield-blvd-1z…","""Q - 56th Ave/ Springfield Blvd""","""56th Avenue""","""Queens""","""No""",3,179,0.51
"""BK-fountain-ave-flatlands-ave-…","""BK - Fountain Ave/Flatlands Av…","""Flatlands Avenue""","""Brooklyn""","""No""",8,563,0.43
"""SI-haven-ave-adam-ave-1qijg0""","""SI - Haven Ave/ Adam Ave""","""Adams Avenue""","""Staten Island""","""No""",11,770,0.43
"""BK-hoyt-st-5th-st-007sk0""","""BK - Hoyt St/5th St""","""Hoyt Street""","""Brooklyn""","""No""",25,1981,0.38
"""BK-2nd-ave-49th-st-2l8qvc""","""BK - 2nd Ave/49th St""","""2nd Avenue""","""Brooklyn""","""No""",2,173,0.35


# 3) Flood Duration
Examine highest total flood duration of any event.

## 3.1 Top 10 Tidally Influenced Sensors by Longest Flood Duration.

In [14]:
sql = """
    SELECT
        sensor_name,
        flood_start_time_est,
        flood_end_time_est,
        duration_mins,
        max_depth_inches,
        tidally_influenced,
        street_name,
        borough
    FROM merged_df
    WHERE tidally_influenced = 'Yes'
    ORDER BY duration_mins DESC
    LIMIT 10
"""

duckdb.sql(sql).pl()

sensor_name,flood_start_time_est,flood_end_time_est,duration_mins,max_depth_inches,tidally_influenced,street_name,borough
str,"datetime[μs, America/New_York]","datetime[μs, America/New_York]",f64,f64,str,str,str
"""SI - Quincy Ave/Iona St""",2025-02-16 01:31:03 EST,2025-02-18 11:37:03 EST,3485.99,3.31,"""Yes""","""Quincy Avenue, Comanche Avenue""","""Staten Island"""
"""SI - Quincy Ave/Iona St""",2024-04-02 14:36:42 EDT,2024-04-04 14:42:50 EDT,2886.13,3.98,"""Yes""","""Quincy Avenue, Comanche Avenue""","""Staten Island"""
"""BX - Cross St/Minnieford Ave""",2024-04-02 15:17:22 EDT,2024-04-04 12:16:11 EDT,2698.82,2.52,"""Yes""","""Minnieford Avenue""","""Bronx"""
"""SI - Quincy Ave/Iona St""",2023-10-20 16:22:29 EDT,2023-10-22 12:10:08 EDT,2627.65,3.46,"""Yes""","""Quincy Avenue, Comanche Avenue""","""Staten Island"""
"""SI - Quincy Ave/Iona St""",2023-09-29 03:53:34 EDT,2023-09-30 15:59:42 EDT,2166.14,5.59,"""Yes""","""Quincy Avenue, Comanche Avenue""","""Staten Island"""
"""SI - Quincy Ave/Iona St""",2025-05-22 05:44:09 EDT,2025-05-23 14:10:54 EDT,1946.75,1.97,"""Yes""","""Quincy Avenue, Comanche Avenue""","""Staten Island"""
"""Q - Russell St 2""",2022-10-03 13:48:55 EDT,2022-10-04 20:17:31 EDT,1828.59,7.48,"""Yes""","""Russell Street""","""Queens"""
"""BX - Cross St/Minnieford Ave""",2024-03-06 15:56:28 EST,2024-03-07 20:18:55 EST,1702.46,2.05,"""Yes""","""Minnieford Avenue""","""Bronx"""
"""BX - Cross St/Minnieford Ave""",2024-01-28 06:35:01 EST,2024-01-29 09:36:54 EST,1621.88,1.5,"""Yes""","""Minnieford Avenue""","""Bronx"""


## 3.2 Top 10 Non-Tidally Influenced Sensors by Longest Flood Duration.

In [15]:
sql = """
    SELECT
        sensor_name,
        flood_start_time_est,
        flood_end_time_est,
        duration_mins,
        max_depth_inches,
        tidally_influenced,
        street_name,
        borough
    FROM merged_df
    WHERE tidally_influenced = 'No'
    ORDER BY duration_mins DESC
    LIMIT 10
"""

duckdb.sql(sql).pl()

sensor_name,flood_start_time_est,flood_end_time_est,duration_mins,max_depth_inches,tidally_influenced,street_name,borough
str,"datetime[μs, America/New_York]","datetime[μs, America/New_York]",f64,f64,str,str,str
"""BK - Hoyt St/5th St""",2024-03-06 16:10:53 EST,2024-03-07 10:38:14 EST,1107.36,4.25,"""No""","""Hoyt Street""","""Brooklyn"""
"""BK - Hoyt St/5th St""",2024-08-06 18:45:32 EDT,2024-08-07 12:31:28 EDT,1065.93,4.17,"""No""","""Hoyt Street""","""Brooklyn"""
"""BK - Hoyt St/5th St""",2023-12-18 01:08:45 EST,2023-12-18 18:14:08 EST,1025.38,5.59,"""No""","""Hoyt Street""","""Brooklyn"""
"""BX - 1st St/Avenue A""",2025-04-26 19:17:20 EDT,2025-04-27 07:55:34 EDT,758.24,6.18,"""No""","""Avenue A""","""Bronx"""
"""BX - 1st St/Avenue A""",2025-03-05 18:41:55 EST,2025-03-06 06:56:39 EST,734.74,4.29,"""No""","""Avenue A""","""Bronx"""
"""BX - Tibbett Ave/W 234th St""",2025-12-02 10:00:42 EST,2025-12-02 21:00:40 EST,659.97,1.14,"""No""","""Tibbett Avenue""","""Bronx"""
"""BX - Webster Ave/E 166th St""",2025-06-28 21:07:15 EDT,2025-06-29 07:36:13 EDT,628.97,3.43,"""No""","""Webster Avenue""","""Bronx"""
"""SI - Haven Ave/ Adam Ave""",2024-03-23 12:32:44 EDT,2024-03-23 21:55:19 EDT,562.58,5.28,"""No""","""Adams Avenue""","""Staten Island"""
"""BX - Tibbett Ave/W 234th St""",2025-05-31 01:52:23 EDT,2025-05-31 11:13:00 EDT,560.61,1.38,"""No""","""Tibbett Avenue""","""Bronx"""


## 3.3 Top 10 Sensors by Flood Duration for Events Exceeding 12 inches.
We restrict the rankings to only large flooding events (i.e. 12 inches).

In [16]:
sql = """
    SELECT
        sensor_name,
        flood_start_time_est,
        flood_end_time_est,
        duration_mins,
        max_depth_inches,
        tidally_influenced,
        street_name,
        borough
    FROM merged_df
    WHERE tidally_influenced = 'Yes'
        AND max_depth_inches >= 12
    ORDER BY duration_mins DESC
    LIMIT 10
"""

duckdb.sql(sql).pl()

sensor_name,flood_start_time_est,flood_end_time_est,duration_mins,max_depth_inches,tidally_influenced,street_name,borough
str,"datetime[μs, America/New_York]","datetime[μs, America/New_York]",f64,f64,str,str,str
"""Q - Beach 84 St""",2022-01-01 18:14:07 EST,2022-01-02 11:06:22 EST,1012.26,22.44,"""Yes""","""Beach 84th Street""","""Queens"""
"""Q - Beach 84 St""",2023-09-29 06:51:28 EDT,2023-09-29 23:12:18 EDT,980.84,16.65,"""Yes""","""Beach 84th Street""","""Queens"""
"""BX - Ditmars St/Hunter Ave 2""",2024-04-03 12:04:19 EDT,2024-04-04 02:34:36 EDT,870.27,34.65,"""Yes""","""Ditmars Street""","""Bronx"""
"""Q - Davenport Ct 1""",2022-12-23 05:24:23 EST,2022-12-23 16:29:21 EST,664.96,40.87,"""Yes""","""Davenport Court""","""Queens"""
"""Q - Davenport Ct 1""",2023-09-29 07:11:07 EDT,2023-09-29 17:46:33 EDT,635.44,15.43,"""Yes""","""Davenport Court""","""Queens"""
"""BX - Ditmars St/Hunter Ave 2""",2024-08-06 17:13:03 EDT,2024-08-07 02:55:34 EDT,582.53,15.75,"""Yes""","""Ditmars Street""","""Bronx"""
"""Q - Davenport Ct 1""",2024-01-13 08:05:35 EST,2024-01-13 17:19:03 EST,553.46,34.69,"""Yes""","""Davenport Court""","""Queens"""
"""Q - Davenport Ct 1""",2024-03-09 18:52:22 EST,2024-03-10 05:00:51 EDT,548.48,20.71,"""Yes""","""Davenport Court""","""Queens"""
"""Q - Beach 72nd St/Almeda Ave""",2023-09-29 10:30:26 EDT,2023-09-29 19:35:04 EDT,544.65,12.24,"""Yes""","""Beach 72nd Street""","""Queens"""


## 3.4 Top 10 Sensors by Duration of Flooding Greater Than 12 Inches
Here, we rank sensors and events by total flood duration that's above 12 inches.

In [17]:
sql = """
    SELECT
        sensor_name,
        flood_start_time_est,
        flood_end_time_est,
        duration_mins,
        duration_above_12_inches_mins,
        max_depth_inches,
        tidally_influenced,
        street_name,
        borough
    FROM merged_df
    ORDER BY duration_above_12_inches_mins DESC
    LIMIT 10
"""

duckdb.sql(sql).pl()

sensor_name,flood_start_time_est,flood_end_time_est,duration_mins,duration_above_12_inches_mins,max_depth_inches,tidally_influenced,street_name,borough
str,"datetime[μs, America/New_York]","datetime[μs, America/New_York]",f64,f64,f64,str,str,str
"""Q - Davenport Ct 1""",2022-12-23 05:24:23 EST,2022-12-23 16:29:21 EST,664.96,516.27,40.87,"""Yes""","""Davenport Court""","""Queens"""
"""Q - Davenport Ct 1""",2023-09-29 07:11:07 EDT,2023-09-29 17:46:33 EDT,635.44,445.59,15.43,"""Yes""","""Davenport Court""","""Queens"""
"""BX - Ditmars St/Hunter Ave 2""",2024-04-03 12:04:19 EDT,2024-04-04 02:34:36 EDT,870.27,413.82,34.65,"""Yes""","""Ditmars Street""","""Bronx"""
"""Q - Davenport Ct 1""",2024-01-13 08:05:35 EST,2024-01-13 17:19:03 EST,553.46,393.28,34.69,"""Yes""","""Davenport Court""","""Queens"""
"""Q - Davenport Ct 1""",2024-03-09 18:52:22 EST,2024-03-10 05:00:51 EDT,548.48,386.02,20.71,"""Yes""","""Davenport Court""","""Queens"""
"""Q - Davenport Ct 1""",2024-04-04 02:53:17 EDT,2024-04-04 11:50:59 EDT,537.7,378.5,20.83,"""Yes""","""Davenport Court""","""Queens"""
"""Q - Davenport Ct 1""",2024-03-10 07:44:49 EDT,2024-03-10 15:32:08 EDT,467.32,325.8,26.06,"""Yes""","""Davenport Court""","""Queens"""
"""BX - Ditmars St/Hunter Ave 2""",2024-08-06 17:13:03 EDT,2024-08-07 02:55:34 EDT,582.53,314.1,15.75,"""Yes""","""Ditmars Street""","""Bronx"""
"""Q - Davenport Ct 1""",2024-01-10 05:39:07 EST,2024-01-10 13:13:49 EST,454.7,293.19,23.94,"""Yes""","""Davenport Court""","""Queens"""


# 4) Average Flood Duration per Event
When flooding occurs at this sensor, how long does a typical event last? This statistic tells you just that.

## 4.1 Top 10 Sensors by Average Flood Duration.

In [18]:
rel = duckdb.sql("""
    SELECT
        sensor_id,
        sensor_name,
        street_name,
        borough,
        tidally_influenced,
        COUNT(sensor_id) AS flood_events,
        SUM(duration_mins) AS duration_mins
    FROM merged_df
    GROUP BY ALL
""")

sql = """
    SELECT
        *,
        duration_mins / flood_events / 60. AS average_duration_hrs
    FROM rel
    ORDER BY average_duration_hrs DESC
    LIMIT 10
"""

duckdb.sql(sql).pl()

sensor_id,sensor_name,street_name,borough,tidally_influenced,flood_events,duration_mins,average_duration_hrs
str,str,str,str,str,i64,f64,f64
"""BX-cross-st-minnieford-ave-1kw…","""BX - Cross St/Minnieford Ave""","""Minnieford Avenue""","""Bronx""","""Yes""",9,11437.37,21.180315
"""SI-quincy-ave-iona-st-1deh00""","""SI - Quincy Ave/Iona St""","""Quincy Avenue, Comanche Avenue""","""Staten Island""","""Yes""",34,32593.16,15.977039
"""BX-1st-st-avenue-a-1zby90""","""BX - 1st St/Avenue A""","""Avenue A""","""Bronx""","""No""",4,1899.87,7.916125
"""BX-ditmars-st-hunter-ave-1hczk…","""BX - Ditmars St/Hunter Ave 1""","""Ditmars Street""","""Bronx""","""Yes""",1,439.45,7.324167
"""Q-beach-66th-st-thursby-ave-1a…","""Q - Beach 66th St/Thursby Ave""","""Beach 66th Street""","""Queens""","""Yes""",1,409.64,6.827333
"""SI-lenevar-ave--drumgoole-rd-w…","""SI - Lenevar Ave & Drumgoole R…","""Lenevar Avenue""","""Staten Island""","""No""",1,330.67,5.511167
"""BK-malcolm-x-blvd-fulton-st-2m…","""BK - Fulton St/ Malcom X Blvd""","""Fulton Street""","""Brooklyn""","""No""",1,328.97,5.482833
"""BK-9th-st-smith-st-0etvg0""","""BK - 9th St/Smith St""","""9th Street""","""Brooklyn""","""No""",2,632.56,5.271333
"""Q-davenport-ct-2-07znk0""","""Q - Davenport Ct 2""","""Davenport Court""","""Queens""","""Yes""",13,3869.63,4.961064


## 4.2 Top 10 Sensors by Average Flood Duration for Events Exceeding 12 inches.

In [19]:
rel = duckdb.sql("""
    SELECT
        sensor_id,
        sensor_name,
        street_name,
        borough,
        tidally_influenced,
        COUNT(sensor_id) AS flood_events,
        SUM(duration_above_12_inches_mins) AS duration_above_12_inches_mins
    FROM merged_df
    WHERE duration_above_12_inches_mins > 0
    GROUP BY ALL
""")

sql = """
    SELECT
        *,
        duration_above_12_inches_mins / flood_events / 60. AS average_duration_hrs
    FROM rel
    ORDER BY average_duration_hrs DESC
    LIMIT 10
"""

duckdb.sql(sql).pl()

sensor_id,sensor_name,street_name,borough,tidally_influenced,flood_events,duration_above_12_inches_mins,average_duration_hrs
str,str,str,str,str,i64,f64,f64
"""Q-davenport-ct-1-07zks0""","""Q - Davenport Ct 1""","""Davenport Court""","""Queens""","""Yes""",18,4336.86,4.015611
"""Q-davenport-ct-2-07znk0""","""Q - Davenport Ct 2""","""Davenport Court""","""Queens""","""Yes""",1,208.42,3.473667
"""BX-ditmars-st-hunter-ave-1kwrk…","""BX - Ditmars St/Hunter Ave 2""","""Ditmars Street""","""Bronx""","""Yes""",11,1828.02,2.769727
"""BK-9th-st-smith-st-0etvg0""","""BK - 9th St/Smith St""","""9th Street""","""Brooklyn""","""No""",2,294.42,2.4535
"""BK-lee-ave-middleton-st-19qok0""","""BK - Lee Ave/Middleton St""","""Lee Avenue""","""Brooklyn""","""No""",2,241.34,2.011167
"""Q-161st-ave-99th-st-1a5q80""","""Q - 161st Ave/99th St""","""161st Avenue, Grimm Avenue""","""Queens""","""Yes""",1,120.25,2.004167
"""Q-beach-43rd-st-15qgw0""","""Q - Beach 43rd St""","""Beach 43rd Street""","""Queens""","""Yes""",2,217.7,1.814167
"""Q-brookville-blvd-rockaway-blv…","""Q - Brookville Blvd/ Snake Rd …","""Brookville Boulevard""","""Queens""","""Yes""",1,106.4,1.773333
"""Q-beach-72nd-st-almeda-ave-1a5…","""Q - Beach 72nd St/Almeda Ave""","""Beach 72nd Street""","""Queens""","""Yes""",2,204.97,1.708083


# 5) Duration of Flood Events by Total Sensor Duration
Instead of number of events, we normalize by days active of the sensor. This helps us answer questions such as , "On average, how many hours per month does this sensor flood?" What proportion of time is this location flooded?

## Normalize total minutes by days active

In [20]:
deployment_days = duckdb.sql("""
    SELECT
        sensor_id,
        date_diff('day', date_installed, current_date()) AS deployment_days
    FROM metadata_df
""")

flood_events = duckdb.sql("""
    SELECT
        sensor_id,
        sensor_name,
        street_name,
        borough,
        tidally_influenced,
        SUM(duration_mins) AS duration_mins,
    FROM merged_df
    GROUP BY ALL
""")

## 5.1 Total Flood Duration per Month by Tidally Influenced Sensors.

In [21]:
sql = """
    SELECT
        *,
        ROUND((duration_mins / 60. * 30) / deployment_days, 2) AS average_hours_per_month
    FROM flood_events
    JOIN deployment_days USING (sensor_id)
    WHERE tidally_influenced = 'Yes'
    ORDER BY average_hours_per_month DESC
    LIMIT 10
"""

duckdb.sql(sql).pl()

sensor_id,sensor_name,street_name,borough,tidally_influenced,duration_mins,deployment_days,average_hours_per_month
str,str,str,str,str,f64,i64,f64
"""BX-ditmars-st-hunter-ave-1kwrk…","""BX - Ditmars St/Hunter Ave 2""","""Ditmars Street""","""Bronx""","""Yes""",50596.61,879,28.78
"""Q-beach-84-st-0me680""","""Q - Beach 84 St""","""Beach 84th Street""","""Queens""","""Yes""",66414.81,1550,21.42
"""SI-quincy-ave-iona-st-1deh00""","""SI - Quincy Ave/Iona St""","""Quincy Avenue, Comanche Avenue""","""Staten Island""","""Yes""",32593.16,1025,15.9
"""Q-brookville-blvd-rockaway-blv…","""Q - Brookville Blvd/ Snake Rd …","""Brookville Boulevard""","""Queens""","""Yes""",9833.24,515,9.55
"""Q-beach-84th-st-beach-channel-…","""Q - Beach 84 St (2)""","""Beach 84th Street""","""Queens""","""Yes""",2468.9,131,9.42
"""Q-russell-st-1-07zqc0""","""Q - Russell St 1""","""Russell Street""","""Queens""","""Yes""",26554.32,1830,7.26
"""Q-brookville-blvd-rockaway-blv…","""Q - Brookville Blvd/ Snake Rd …","""Brookville Boulevard""","""Queens""","""Yes""",7037.61,515,6.83
"""BX-cross-st-minnieford-ave-1kw…","""BX - Cross St/Minnieford Ave""","""Minnieford Avenue""","""Bronx""","""Yes""",11437.37,879,6.51
"""Q-brookville-blvd-149th-ave-3-…","""Q - Brookville Blvd/ Snake Rd …","""Brookville Boulevard""","""Queens""","""Yes""",6188.23,515,6.01


## 5.2 Total Flood Duration per Month by Non-Tidally Influenced Sensors.

In [22]:
sql = """
    SELECT
        *,
        ROUND((duration_mins / 60. * 30) / deployment_days, 2) AS average_hours_per_month
    FROM flood_events
    JOIN deployment_days USING (sensor_id)
    WHERE tidally_influenced = 'No'
    ORDER BY average_hours_per_month DESC
    LIMIT 10
"""

duckdb.sql(sql).pl()

sensor_id,sensor_name,street_name,borough,tidally_influenced,duration_mins,deployment_days,average_hours_per_month
str,str,str,str,str,f64,i64,f64
"""BX-w-234th-st-tibbett-ave-2cak…","""BX - Tibbett Ave/W 234th St""","""Tibbett Avenue""","""Bronx""","""No""",3196.52,347,4.61
"""BX-1st-st-avenue-a-1zby90""","""BX - 1st St/Avenue A""","""Avenue A""","""Bronx""","""No""",1899.87,599,1.59
"""BX-webster-ave-e-166th-st-2151…","""BX - Webster Ave/E 166th St""","""Webster Avenue""","""Bronx""","""No""",1783.25,564,1.58
"""Q-56th-ave-springfield-blvd-1z…","""Q - 56th Ave/ Springfield Blvd""","""56th Avenue""","""Queens""","""No""",534.96,179,1.49
"""BK-hoyt-st-5th-st-007sk0""","""BK - Hoyt St/5th St""","""Hoyt Street""","""Brooklyn""","""No""",5548.34,1981,1.4
"""SI-haven-ave-adam-ave-1qijg0""","""SI - Haven Ave/ Adam Ave""","""Adams Avenue""","""Staten Island""","""No""",2070.42,770,1.34
"""BK-malcolm-x-blvd-fulton-st-2m…","""BK - Fulton St/ Malcom X Blvd""","""Fulton Street""","""Brooklyn""","""No""",328.97,150,1.1
"""BK-2nd-ave-49th-st-2l8qvc""","""BK - 2nd Ave/49th St""","""2nd Avenue""","""Brooklyn""","""No""",232.0,173,0.67
"""BK-neptune-ave-w-35th-st-1xvfk…","""BK - Neptune Ave W 35th St""","""West 35th Street""","""Brooklyn""","""No""",823.72,627,0.66


# 6) Quickest Time to X Inches

In [23]:
def time_to_threshold(
    df,
    threshold_inches: float,
    top_n: int = 10,
    ts_sec_col="flood_profile_time_secs",
    ts_in_col="flood_profile_depth_inches",
    max_col="max_depth_inches",
    name_col="sensor_name",
    start_col="flood_start_time_est",
    end_col="flood_end_time_est",
):

    if not isinstance(df, pl.DataFrame):
        df = pl.from_pandas(df)

    if df.is_empty():
        return pl.DataFrame()

    df = df.with_row_index("event_id")

    # Parse list columns
    df = df.with_columns([
        pl.col(ts_sec_col).map_elements(
            lambda x: ast.literal_eval(x) if isinstance(x, str) else x,
            return_dtype=pl.List(pl.Float64),
        ).alias("sec"),

        pl.col(ts_in_col).map_elements(
            lambda x: ast.literal_eval(x) if isinstance(x, str) else x,
            return_dtype=pl.List(pl.Float64),
        ).alias("depth"),
    ])

    # Filter eligible events
    df = df.filter(
        pl.col("sec").is_not_null()
        & pl.col("depth").is_not_null()
        & (pl.col(max_col) >= threshold_inches)
    )

    if df.is_empty():
        return pl.DataFrame()

    # Explode time series
    long = (
        df.select(
            "event_id",
            name_col,
            start_col,
            end_col,
            max_col,
            "sec",
            "depth",
        )
        .explode(["sec", "depth"])
        .with_columns([
            pl.col("sec").cast(pl.Float64),
            pl.col("depth").cast(pl.Float64),
        ])
        .drop_nulls(["sec", "depth"])
    )

    # Normalize time and clamp depth
    long = (
        long
        .with_columns(
            (pl.col("sec") - pl.first("sec").over("event_id")).alias("rel_sec")
        )
        .with_columns(
            pl.when(pl.col("depth") < 0)
            .then(0.0)
            .otherwise(pl.col("depth"))
            .alias("depth")
        )
        .sort(["event_id", "rel_sec"])
    )

    # store full time series
    series_data = (
        long
        .group_by("event_id")
        .agg([
            pl.col("rel_sec").implode().alias("x_values"),
            pl.col("depth").implode().alias("y_values"),
        ])
    )

    # add previous point for interpolation
    long = long.with_columns([
        pl.col("depth").shift(1).over("event_id").alias("prev_depth"),
        pl.col("rel_sec").shift(1).over("event_id").alias("prev_sec"),
        pl.first("depth").over("event_id").alias("start_depth"),
    ])

    # detect crossings
    crossings = long.filter(
        (pl.col("prev_depth") < threshold_inches) &
        (pl.col("depth") >= threshold_inches)
    )

    # interpolate crossing time
    crossings = crossings.with_columns(
        (
            pl.col("prev_sec")
            + (threshold_inches - pl.col("prev_depth"))
            / (pl.col("depth") - pl.col("prev_depth"))
            * (pl.col("rel_sec") - pl.col("prev_sec"))
        ).alias("time_to_thresh_sec")
    )

    # collapse to one row per event
    crossings = (
        crossings
        .group_by("event_id")
        .agg([
            pl.first("time_to_thresh_sec"),
            pl.first("start_depth"),
            pl.first(name_col).alias("name"),
            pl.first(start_col).alias("start"),
            pl.first(end_col).alias("end"),
            pl.first(max_col).alias("depth_max_inches"),
        ])
    )

    if crossings.is_empty():
        return pl.DataFrame()

    crossings = crossings.join(series_data, on="event_id", how="left")

    crossings = crossings.with_columns([
        pl.lit(threshold_inches).alias("threshold_inches"),
        pl.col("time_to_thresh_sec").round(2),
        (pl.col("time_to_thresh_sec") / 60).round(2).alias("time_to_thresh_minutes"),
        (
            (threshold_inches - pl.col("start_depth"))
            / (pl.col("time_to_thresh_sec") / 60)
        ).round(2).alias("rate_to_thresh_in_per_min"),
    ])

    return (
        crossings
        .sort("time_to_thresh_sec")
        .head(top_n)
        .select([
            "name",
            "depth_max_inches",
            "start",
            "end",
            "threshold_inches",
            "time_to_thresh_sec",
            "time_to_thresh_minutes",
            "rate_to_thresh_in_per_min",
            "x_values",
            "y_values",
        ])
    )

## 6.1 Quickest Time to 12 Inches.

In [24]:
threshold_inches = 12

time_to_threshold(
    df=merged_df,
    threshold_inches=threshold_inches
)

name,depth_max_inches,start,end,threshold_inches,time_to_thresh_sec,time_to_thresh_minutes,rate_to_thresh_in_per_min,x_values,y_values
str,f64,"datetime[μs, America/New_York]","datetime[μs, America/New_York]",i32,f64,f64,f64,list[f64],list[f64]
"""BK - Carroll St/4th Av""",31.3,2022-09-13 04:35:09 EDT,2022-09-13 05:21:25 EDT,12,136.93,2.28,5.26,"[0.0, 63.0, … 2777.0]","[0.0, 3.46, … 0.0]"
"""BK - Carroll St/4th Av""",35.94,2023-09-29 07:53:39 EDT,2023-09-29 10:02:50 EDT,12,175.25,2.92,4.11,"[0.0, 63.0, … 7751.0]","[0.0, 1.22, … 0.0]"
"""BK - Carroll St/4th Av""",35.59,2021-09-01 21:27:16 EDT,2021-09-01 23:11:26 EDT,12,209.05,3.48,3.44,"[0.0, 312.0, … 6250.0]","[0.0, 17.91, … 0.0]"
"""Q - 204th St/ 100th Ave""",23.35,2025-10-30 16:04:44 EDT,2025-10-30 16:47:45 EDT,12,234.44,3.91,3.07,"[0.0, 60.0, … 2581.0]","[0.0, 0.98, … 0.0]"
"""BK - Greene Ave / Grand Ave""",17.6,2025-10-30 15:00:01 EDT,2025-10-30 16:02:01 EDT,12,299.87,5.0,2.4,"[0.0, 180.0, … 3720.0]","[0.0, 6.1, … 0.0]"
"""SI - Catherine Ct/Jewett Ave""",21.34,2025-07-31 15:40:39 EDT,2025-07-31 16:28:39 EDT,12,385.49,6.42,1.87,"[0.0, 60.0, … 2880.0]","[0.0, 3.35, … 0.0]"
"""BK - Carroll St/4th Av""",26.85,2021-08-22 00:05:05 EDT,2021-08-22 00:51:45 EDT,12,411.51,6.86,1.75,"[0.0, 311.0, … 2800.0]","[0.0, 8.82, … 0.0]"
"""Q - Liberty Ave/ 183rd St""",19.96,2025-10-30 15:58:49 EDT,2025-10-30 16:45:48 EDT,12,427.09,7.12,1.69,"[0.0, 120.0, … 2819.0]","[0.0, 1.26, … 0.0]"
"""BK - Carroll St/4th Av""",28.62,2021-08-21 21:19:06 EDT,2021-08-21 22:21:20 EDT,12,515.28,8.59,1.4,"[0.0, 311.0, … 3734.0]","[0.0, 5.04, … 0.0]"


## 6.2 Quickest Time to 24 Inches.

In [25]:
threshold_inches = 24

time_to_threshold(
    df=merged_df,
    threshold_inches=threshold_inches, 
)

name,depth_max_inches,start,end,threshold_inches,time_to_thresh_sec,time_to_thresh_minutes,rate_to_thresh_in_per_min,x_values,y_values
str,f64,"datetime[μs, America/New_York]","datetime[μs, America/New_York]",i32,f64,f64,f64,list[f64],list[f64]
"""BK - Carroll St/4th Av""",31.3,2022-09-13 04:35:09 EDT,2022-09-13 05:21:25 EDT,24,309.13,5.15,4.66,"[0.0, 63.0, … 2777.0]","[0.0, 3.46, … 0.0]"
"""BK - Carroll St/4th Av""",35.59,2021-09-01 21:27:16 EDT,2021-09-01 23:11:26 EDT,24,479.61,7.99,3.0,"[0.0, 312.0, … 6250.0]","[0.0, 17.91, … 0.0]"
"""BK - Carroll St/4th Av""",35.94,2023-09-29 07:53:39 EDT,2023-09-29 10:02:50 EDT,24,619.74,10.33,2.32,"[0.0, 63.0, … 7751.0]","[0.0, 1.22, … 0.0]"
"""BK - Carroll St/4th Av""",26.85,2021-08-22 00:05:05 EDT,2021-08-22 00:51:45 EDT,24,889.0,14.82,1.62,"[0.0, 311.0, … 2800.0]","[0.0, 8.82, … 0.0]"
"""BK - Carroll St/4th Av""",28.62,2021-08-21 21:19:06 EDT,2021-08-21 22:21:20 EDT,24,1135.04,18.92,1.27,"[0.0, 311.0, … 3734.0]","[0.0, 5.04, … 0.0]"
"""BK - 9th St/Smith St""",25.35,2023-09-29 07:54:23 EDT,2023-09-29 13:25:16 EDT,24,1922.5,32.04,0.75,"[0.0, 63.0, … 19853.0]","[0.0, 3.07, … 0.51]"
"""BK - Wallabout St/Throop Ave""",25.08,2023-09-29 08:05:49 EDT,2023-09-29 11:03:49 EDT,24,2372.0,39.53,0.61,"[0.0, 63.0, … 10680.0]","[0.0, 1.14, … 0.0]"
"""Q - Davenport Ct 1""",40.87,2022-12-23 05:24:23 EST,2022-12-23 16:29:21 EST,24,2381.93,39.7,0.6,"[0.0, 63.0, … 39898.0]","[0.0, 0.79, … 0.0]"
"""Q - Davenport Ct 1""",26.06,2024-03-10 07:44:49 EDT,2024-03-10 15:32:08 EDT,24,3687.58,61.46,0.39,"[0.0, 126.0, … 28039.0]","[0.0, 0.59, … 0.0]"


## 6.3 Quickest Time to 12 Inches for Tidal vs. Non Tidal Sensors.

In [26]:
threshold_inches = 12

# non-tidal sensors
non_tidal_df = merged_df.filter(pl.col("tidally_influenced") == 'No')

time_to_threshold(
    df=non_tidal_df,
    threshold_inches=threshold_inches
)

name,depth_max_inches,start,end,threshold_inches,time_to_thresh_sec,time_to_thresh_minutes,rate_to_thresh_in_per_min,x_values,y_values
str,f64,"datetime[μs, America/New_York]","datetime[μs, America/New_York]",i32,f64,f64,f64,list[f64],list[f64]
"""BK - Carroll St/4th Av""",31.3,2022-09-13 04:35:09 EDT,2022-09-13 05:21:25 EDT,12,136.93,2.28,5.26,"[0.0, 63.0, … 2777.0]","[0.0, 3.46, … 0.0]"
"""BK - Carroll St/4th Av""",35.94,2023-09-29 07:53:39 EDT,2023-09-29 10:02:50 EDT,12,175.25,2.92,4.11,"[0.0, 63.0, … 7751.0]","[0.0, 1.22, … 0.0]"
"""BK - Carroll St/4th Av""",35.59,2021-09-01 21:27:16 EDT,2021-09-01 23:11:26 EDT,12,209.05,3.48,3.44,"[0.0, 312.0, … 6250.0]","[0.0, 17.91, … 0.0]"
"""Q - 204th St/ 100th Ave""",23.35,2025-10-30 16:04:44 EDT,2025-10-30 16:47:45 EDT,12,234.44,3.91,3.07,"[0.0, 60.0, … 2581.0]","[0.0, 0.98, … 0.0]"
"""BK - Greene Ave / Grand Ave""",17.6,2025-10-30 15:00:01 EDT,2025-10-30 16:02:01 EDT,12,299.87,5.0,2.4,"[0.0, 180.0, … 3720.0]","[0.0, 6.1, … 0.0]"
"""SI - Catherine Ct/Jewett Ave""",21.34,2025-07-31 15:40:39 EDT,2025-07-31 16:28:39 EDT,12,385.49,6.42,1.87,"[0.0, 60.0, … 2880.0]","[0.0, 3.35, … 0.0]"
"""BK - Carroll St/4th Av""",26.85,2021-08-22 00:05:05 EDT,2021-08-22 00:51:45 EDT,12,411.51,6.86,1.75,"[0.0, 311.0, … 2800.0]","[0.0, 8.82, … 0.0]"
"""Q - Liberty Ave/ 183rd St""",19.96,2025-10-30 15:58:49 EDT,2025-10-30 16:45:48 EDT,12,427.09,7.12,1.69,"[0.0, 120.0, … 2819.0]","[0.0, 1.26, … 0.0]"
"""BK - Carroll St/4th Av""",28.62,2021-08-21 21:19:06 EDT,2021-08-21 22:21:20 EDT,12,515.28,8.59,1.4,"[0.0, 311.0, … 3734.0]","[0.0, 5.04, … 0.0]"


In [27]:
threshold_inches = 12

# tidally influenced sensors
tidal_df = merged_df.filter(pl.col("tidally_influenced") == 'Yes')

time_to_threshold(
    df=tidal_df,
    threshold_inches=threshold_inches
)

name,depth_max_inches,start,end,threshold_inches,time_to_thresh_sec,time_to_thresh_minutes,rate_to_thresh_in_per_min,x_values,y_values
str,f64,"datetime[μs, America/New_York]","datetime[μs, America/New_York]",i32,f64,f64,f64,list[f64],list[f64]
"""Q - Beach 43rd St""",23.43,2022-12-23 06:07:36 EST,2022-12-23 09:57:30 EST,12,1074.15,17.9,0.67,"[0.0, 252.0, … 13795.0]","[0.0, 4.06, … 0.0]"
"""Q - 1st St/104th St""",16.65,2024-03-10 07:57:07 EDT,2024-03-10 13:01:30 EDT,12,1078.71,17.98,0.67,"[0.0, 64.0, … 18264.0]","[0.0, 0.63, … 0.0]"
"""Q - Beach 43rd St""",18.46,2024-01-13 09:33:00 EST,2024-01-13 12:16:18 EST,12,1249.11,20.82,0.58,"[0.0, 189.0, … 9798.0]","[0.0, 4.72, … 0.0]"
"""Q - Russell St 2""",31.77,2022-12-23 05:12:13 EST,2022-12-23 11:34:35 EST,12,1320.57,22.01,0.55,"[0.0, 63.0, … 22943.0]","[0.0, 0.59, … 0.0]"
"""Q - Davenport Ct 1""",40.87,2022-12-23 05:24:23 EST,2022-12-23 16:29:21 EST,12,1486.19,24.77,0.48,"[0.0, 63.0, … 39898.0]","[0.0, 0.79, … 0.0]"
"""Q - Russell St 1""",31.34,2022-12-23 05:10:48 EST,2022-12-23 11:04:16 EST,12,1520.35,25.34,0.47,"[0.0, 303.0, … 21208.0]","[0.0, 1.97, … 0.0]"
"""Q - 1st St/104th St""",13.7,2024-04-04 03:00:17 EDT,2024-04-04 08:12:06 EDT,12,1533.49,25.56,0.47,"[0.0, 63.0, … 18709.0]","[0.0, 0.51, … 0.0]"
"""Q - Russell St 1""",17.09,2024-03-10 07:35:18 EDT,2024-03-10 12:29:27 EDT,12,1700.85,28.35,0.42,"[0.0, 63.0, … 17649.0]","[0.0, 1.1, … 0.0]"
"""Q - 1st St/104th St""",12.91,2024-03-09 19:00:51 EST,2024-03-10 01:57:58 EST,12,1867.0,31.12,0.39,"[0.0, 63.0, … 25027.0]","[0.0, 0.98, … 0.0]"
